# 🐶🐱 Dogs vs Cats — Full ML Pipeline
Steps: Setup → Load → Preprocess → Model → Train → Evaluate → Predict

## Step 1 — Install & Import

In [1]:
# !pip install kagglehub tensorflow matplotlib scikit-learn

import os, kagglehub, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
print('TF:', tf.__version__)

ModuleNotFoundError: No module named 'kagglehub'

## Step 2 — Download Dataset

In [ ]:
path = kagglehub.dataset_download('moazeldsokyx/dogs-vs-cats')
print('Dataset path:', path)

# Explore structure
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    if level < 3:
        print('  ' * level + os.path.basename(root) + '/')
        if level == 2: print('  ' * (level+1) + f'{len(files)} files')

NameError: name 'kagglehub' is not defined

: 

## Step 3 — Preprocessing (ImageDataGenerator)

In [ ]:
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

# Find train directory (adjust if structure differs)
train_dir = os.path.join(path, 'train')  # update if needed

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training'
)
val_data = train_gen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation'
)

print('Classes:', train_data.class_indices)
print('Train batches:', len(train_data), '| Val batches:', len(val_data))

## Step 4 — Visualize Sample Images

In [ ]:
imgs, labels = next(train_data)
class_names = {v: k for k, v in train_data.class_indices.items()}

plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 4, i+1)
    plt.imshow(imgs[i])
    plt.title(class_names[int(labels[i])])
    plt.axis('off')
plt.tight_layout(); plt.show()

## Step 5 — Build CNN Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'), layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')   # binary: 0=cat, 1=dog
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Step 6 — Train Model

In [ ]:
cb = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
]

history = model.fit(train_data, validation_data=val_data, epochs=15, callbacks=cb)

## Step 7 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend()

plt.tight_layout(); plt.show()

## Step 8 — Evaluate & Confusion Matrix

In [ ]:
val_data.reset()
y_pred = (model.predict(val_data) > 0.5).astype(int).flatten()
y_true = val_data.classes[:len(y_pred)]

print(classification_report(y_true, y_pred, target_names=['cat', 'dog']))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['cat','dog']).plot(cmap='Blues')
plt.show()

## Step 9 — Predict on a Single Image

In [ ]:
from tensorflow.keras.preprocessing import image as kimage

def predict_image(img_path):
    img = kimage.load_img(img_path, target_size=IMG_SIZE)
    arr = kimage.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    prob = model.predict(arr)[0][0]
    label = 'Dog 🐶' if prob > 0.5 else 'Cat 🐱'
    plt.imshow(img); plt.title(f'{label} ({prob:.2%})'); plt.axis('off'); plt.show()

# Usage — replace with any image path:
# predict_image('/path/to/your/image.jpg')

## Step 10 — Save & Load Model

In [ ]:
model.save('dogs_vs_cats_model.h5')
print('Model saved!')

# To reload:
# model = tf.keras.models.load_model('dogs_vs_cats_model.h5')

## ✅ Pipeline Summary
| Step | Description |
|------|-------------|
| 1 | Install & import libraries |
| 2 | Download dataset via kagglehub |
| 3 | Preprocess with augmentation + split |
| 4 | Visualize sample images |
| 5 | Build CNN model |
| 6 | Train with early stopping |
| 7 | Plot accuracy/loss curves |
| 8 | Evaluate + confusion matrix |
| 9 | Single image prediction |
| 10 | Save/load model |